# 🌌 Dreamer v3 Dream Viewer
Load the trained world model checkpoints and the replay buffer to visualize imagination rollouts (dreams) starting from actual historical states.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
import os
import glob
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Load Dreamer models
from dreamer import Dreamer

# --- Style ------------------------------------------------------------------
display(HTML("""
<style>
  .viewer-root { background:#0d1117; border-radius:12px; padding:16px; }
  .info-box { background:#161b22; border:1px solid #30363d; border-radius:8px;
              padding:10px 14px; font-family:monospace; font-size:12px; color:#8b949e; }
  h1, h2, h3 { font-family: 'Segoe UI', sans-serif; color: #a371f7; }
</style>
"""))
print("UI styles and imports loaded.")

In [ ]:
# --- Configuration & Initialization ----------------------------------------
IMAGE_SIZE = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# We use the exact configuration from policy.py
action_dim = 6
recurrent_dim = 2048 
rows = 40           
cols = 40
latent_dim = rows * cols
buffer_size = 1500000
curiosity_scale = 1.0  

print("Initializing Dreamer...")
dreamer = Dreamer(
    device=device,
    obs_dim=(3, IMAGE_SIZE, IMAGE_SIZE),
    envs=None, # No PyBoy environments needed for visualization!
    action_dim=action_dim,
    recurrent_dim=recurrent_dim,
    rows=rows,
    cols=cols,
    latent_dim=latent_dim,
    number_of_sequences=64, # Batch size needed for encoder view shaping
    steps_per_sequence=64,  # Sequence size needed for encoder view shaping
    seed=1234,
    buffer_size=buffer_size,
    ram_dim=14,
    team_dim=6,
    item_dim=2,
    curiosity_scale=curiosity_scale
)

# Load weights and the replay buffer from checkpoints
dreamer.loadCheckpoints()

# Set all networks to evaluation mode
dreamer.recurrentModel.eval()
dreamer.posteriorNet.eval()
dreamer.priorNet.eval()
dreamer.image_encoder.eval()
dreamer.goal_encoder.eval()
dreamer.team_encoder.eval()
dreamer.item_encoder.eval()
dreamer.actor.eval()
dreamer.decoder.eval()
dreamer.critic.eval()
dreamer.curiosityPredictor.eval()
dreamer.rewardPredictor.eval()

print("Dreamer initialized and checkpoints loaded successfully.")

In [ ]:
# --- Warm-up & Inference Helpers --------------------------------------------

@torch.no_grad()
def get_grounded_states(dreamer, batch_data):
    """
    Feeds a sequence of actual observations, actions, and rewards from the buffer
    through the image/goal/team/item encoders and the posterior GRU to produce
    grounded latent states. This is a non-gradient version of TrainWorldModel.
    """
    obs_flat = batch_data["observations"].flatten(0, 1)
    ram_flat = batch_data["rams"].flatten(0, 1)
    team_flat = batch_data["team_levels"].flatten(0, 1)
    item_flat = batch_data["item_counts"].flatten(0, 1)
    
    encoded_images = dreamer.image_encoder(obs_flat).view(dreamer.number_of_sequences, dreamer.steps_per_sequence, -1)
    encoded_goals = dreamer.goal_encoder(ram_flat).view(dreamer.number_of_sequences, dreamer.steps_per_sequence, -1)   
    encoded_team = dreamer.team_encoder(team_flat / 100.0).view(dreamer.number_of_sequences, dreamer.steps_per_sequence, -1)
    encoded_items = dreamer.item_encoder(item_flat / 10.0).view(dreamer.number_of_sequences, dreamer.steps_per_sequence, -1)
    
    previous_recurrent_state = torch.zeros(dreamer.number_of_sequences, dreamer.recurrent_dim, device=dreamer.device) 
    previous_latent_state = torch.zeros(dreamer.number_of_sequences, dreamer.latent_dim, device=dreamer.device) 
    
    recurrent_states = []
    posteriors = []

    for t in range(1, dreamer.steps_per_sequence): 
        recurrent_state = dreamer.recurrentModel(previous_recurrent_state, previous_latent_state, batch_data["actions"][:, t-1])
        posterior_input = torch.cat((recurrent_state, encoded_images[:, t], encoded_goals[:, t], encoded_team[:, t], encoded_items[:, t]), dim=-1)
        posterior, _ = dreamer.posteriorNet(posterior_input)

        recurrent_states.append(recurrent_state)
        posteriors.append(posterior)

        previous_recurrent_state = recurrent_state
        previous_latent_state = posterior
    
    recurrent_states = torch.stack(recurrent_states, dim=1) 
    posteriors = torch.stack(posteriors, dim=1) 
    full_states = torch.cat((recurrent_states, posteriors), -1) 
    
    # Return shape: (B * (T-1), concatenated_dim)
    return full_states.view(-1, dreamer.concatenated_dim).detach()


@torch.no_grad()
def dream_no_update(dreamer, full_state, horizon=25):
    """
    Rolls out the dreamer's policy forward starting from grounded full states.
    This is a non-gradient version of dreamer.Dream that does not perform
    any backward updates or modify network weights.
    """
    full_states = [full_state.detach()]
    log_probabilities = []
    entropies = []
    actions_stack = [] 

    curr_state = full_state.detach()
    recurrent_state, latent_state = torch.split(curr_state, [dreamer.recurrent_dim, dreamer.latent_dim], -1)

    # --- IMAGINATION LOOP ---
    for i in range(horizon):
        action, logprob, entropy = dreamer.actor(curr_state)
        recurrent_state = dreamer.recurrentModel(recurrent_state, latent_state, action)
        latent_state, _ = dreamer.priorNet(recurrent_state)
        
        curr_state = torch.cat((recurrent_state, latent_state), -1) 
        
        full_states.append(curr_state)
        log_probabilities.append(logprob)
        entropies.append(entropy)
        actions_stack.append(action) 

    full_states = torch.stack(full_states, dim=1)                 
    log_probabilities = torch.stack(log_probabilities, dim=1)     
    entropies = torch.stack(entropies, dim=1)                     
    actions_stack = torch.stack(actions_stack, dim=1)             

    imagined_steps = full_states[:, 1:]  # [B, H, concatenated_dim]
    reward_logits_ensemble = dreamer.rewardPredictor(imagined_steps) # [num_heads, B, H, num_bins]
    
    decoded_rewards = []
    for h in range(dreamer.rewardPredictor.num_heads):
        logits = reward_logits_ensemble[h]
        decoded_rewards.append(dreamer.two_hot.decode(logits).squeeze(-1))
    decoded_rewards = torch.stack(decoded_rewards, dim=0) # [num_heads, B, H]
    mean_rewards = decoded_rewards.mean(dim=0)
    predicted_rewards = mean_rewards

    curiosity_logits = dreamer.curiosityPredictor(imagined_steps)
    predicted_curiosity = dreamer.two_hot.decode(curiosity_logits).squeeze(-1) # [B, H]

    imagined_states = full_states.detach()
    
    critic_logits_all = dreamer.critic(imagined_states)
    online_values = dreamer.two_hot.decode(critic_logits_all).squeeze(-1) # [B, H+1]

    curiosity_critic_logits_all = dreamer.curiosity_critic(imagined_states)
    online_curiosity_values = dreamer.two_hot.decode(curiosity_critic_logits_all).squeeze(-1) # [B, H+1]

    continues = torch.full_like(predicted_rewards, 0.999) 
    lambda_values = dreamer.computeLambdaValues(predicted_rewards, online_values, continues) # [B, H]
    curiosity_lambda_values = dreamer.computeLambdaValues(predicted_curiosity, online_curiosity_values, continues) # [B, H]
    
    denominator = dreamer.dynamicDataNormalizer(lambda_values)
    advantages = (lambda_values - online_values[:, :-1].detach()) / denominator
    
    curiosity_denominator = dreamer.curiosityDynamicDataNormalizer(curiosity_lambda_values)
    curiosity_advantages = (curiosity_lambda_values - online_curiosity_values[:, :-1].detach()) / curiosity_denominator
    
    combined_advantages = (1.0 - dreamer.curiosity_scale) * advantages + dreamer.curiosity_scale * curiosity_advantages

    trajectory_advantages = combined_advantages.sum(dim=1) 
    
    best_dream_idx = torch.argmax(trajectory_advantages).item()
    best_dream_data = (
        trajectory_advantages[best_dream_idx].item(),  
        full_states[best_dream_idx].detach().cpu(),
        predicted_rewards[best_dream_idx].detach().cpu(),
        lambda_values[best_dream_idx].detach().cpu(),       
        actions_stack[best_dream_idx].detach().cpu(),
        combined_advantages[best_dream_idx].detach().cpu(),
        "MaxAdv"                                        
    )

    rand_dream_idx = torch.randint(0, trajectory_advantages.shape[0], (1,)).item()
    rand_dream_data = (
        trajectory_advantages[rand_dream_idx].item(), 
        full_states[rand_dream_idx].detach().cpu(),
        predicted_rewards[rand_dream_idx].detach().cpu(),
        lambda_values[rand_dream_idx].detach().cpu(),       
        actions_stack[rand_dream_idx].detach().cpu(),
        combined_advantages[rand_dream_idx].detach().cpu(),
        "Random"
    )
    
    trajectory_curiosity = predicted_curiosity.sum(dim=1)
    cur_dream_idx = torch.argmax(trajectory_curiosity).item()
    cur_dream_data = (
        trajectory_curiosity[cur_dream_idx].item(), 
        full_states[cur_dream_idx].detach().cpu(),
        predicted_rewards[cur_dream_idx].detach().cpu(),
        lambda_values[cur_dream_idx].detach().cpu(),       
        actions_stack[cur_dream_idx].detach().cpu(),
        combined_advantages[cur_dream_idx].detach().cpu(),
        "MaxCuriosity"
    )
    
    return best_dream_data, rand_dream_data, cur_dream_data

print("Helper functions ready.")

In [ ]:
# --- Sample and Visualize Dreams ---------------------------------------------
%matplotlib inline

# 1. Sample a batch of sequences from the replay buffer
print("Sampling a batch of sequences from the buffer...")
sample = dreamer.sample_batch(
    batchSize=dreamer.number_of_sequences, 
    sequenceSize=dreamer.steps_per_sequence
)

if sample is None:
    raise ValueError("Buffer does not contain enough data to sample from.")

print("Warming up recurrent & posterior states...")
# 2. Feed sequence into encoders and GRU to ground the latent state
full_states = get_grounded_states(dreamer, sample)

# 3. Roll out the imagination (Dream) forward for 25 steps starting from these states
print("Generating dreams...")
best_dream, rand_dream, cur_dream = dream_no_update(dreamer, full_states, horizon=25)

# 4. Visualize each dream using the dreamer's visualize_single_dream method
dreams_to_show = [best_dream, rand_dream, cur_dream]

for metric_val, b_states, b_rewards, b_values, b_actions, b_advantages, label in dreams_to_show:
    if label == "MaxCuriosity":
        title_pref = f"{label} Dream (Total Curiosity: {metric_val:+.4f})"
    else:
        title_pref = f"{label} Dream (Total Adv: {metric_val:+.4f})"
    
    print(f"\n[*] Displaying {label} Dream...")
    dreamer.visualize_single_dream(
        b_states.to(device), 
        b_rewards, 
        b_values, 
        b_actions, 
        b_advantages,
        title_prefix=title_pref
    )